In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')




from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier


from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn import metrics
from sklearn.metrics import roc_curve
from sklearn.metrics import recall_score, confusion_matrix, precision_score, f1_score, accuracy_score, classification_report

from sklearn.ensemble import VotingClassifier

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score 
from sklearn.metrics import f1_score, precision_score, recall_score, fbeta_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from sklearn import feature_selection
from sklearn import model_selection
from sklearn import metrics
from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.metrics import auc, roc_auc_score, roc_curve
from sklearn.metrics import make_scorer, recall_score, log_loss
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
df = pd.read_csv("churn.csv")
df


In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

In [ ]:
df = df.drop(["customerID"], axis = 1)
df.head()


In [ ]:
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df.head()


In [ ]:
df.info()


## Exploratory Data Analysis (EDA)

In [ ]:
df["Churn"].value_counts()

In [ ]:
numRetained = df[df.Churn == 'No'].shape[0]
numChurned = df[df.Churn == 'Yes'].shape[0]

# print the percentage of customers that stayed
print(numRetained/(numRetained + numChurned) * 100,'% of customers stayed in the company')
# peint the percentage of customers that left
print(numChurned/(numRetained + numChurned) * 100, '% of customers left from the company')

In [ ]:
type_ = ["No", "yes"]
fig = make_subplots(rows=1, cols=1)

fig.add_trace(go.Pie(labels=type_, values=df['Churn'].value_counts(), name="Churn"))

# Use `hole` to create a donut-like pie chart
fig.update_traces(hole=.4, hoverinfo="label+percent+name", textfont_size=16)

fig.update_layout(
    title_text="Churn Distributions",
    # Add annotations in the center of the donut pies.
    annotations=[dict(text='Churn', x=0.5, y=0.5, font_size=20, showarrow=False)])
fig.show()

In [ ]:
df.Churn[df.Churn == "No"].groupby(by = df.gender).count()

In [ ]:
df.Churn[df.Churn == "Yes"].groupby(by = df.gender).count()

In [ ]:
plt.figure(figsize=(6, 6))
labels =["Churn: Yes","Churn:No"]
values = [1869,5163]
labels_gender = ["F","M","F","M"]
sizes_gender = [939,930 , 2544,2619]
colors = ['#ff6666', '#66b3ff']
colors_gender = ['#c2c2f0','#ffb3e6', '#c2c2f0','#ffb3e6']
explode = (0.3,0.3) 
explode_gender = (0.1,0.1,0.1,0.1)
textprops = {"fontsize":15}
#Plot
plt.pie(values, labels=labels,autopct='%1.1f%%',pctdistance=1.08, labeldistance=0.8,colors=colors, startangle=90,frame=True, explode=explode,radius=10, textprops =textprops, counterclock = True, )
plt.pie(sizes_gender,labels=labels_gender,colors=colors_gender,startangle=90, explode=explode_gender,radius=7, textprops =textprops, counterclock = True, )
#Draw circle
centre_circle = plt.Circle((0,0),5,color='black', fc='white',linewidth=0)
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

plt.title('Churn Distribution w.r.t Gender: Male(M), Female(F)', fontsize=15, y=1.1)

# show plot 
 
plt.axis('equal')
plt.tight_layout()
plt.show()

### Churn is almost equally distributed across male and female customers, indicating that gender is not a significant factor in predicting churn.

###

In [ ]:
fig = px.histogram(df, x="Churn", color = "Contract", barmode = "group", title = "<b>Customer contract distribution<b>")
fig.update_layout(width=700, height=500, bargap=0.2)
fig.show()

### Month-to-month customers show the highest churn, while two-year contract customers have the lowest churn, indicating that contract length is a strong retention factor.

### 

In [ ]:
labels = df['PaymentMethod'].unique()
values = df['PaymentMethod'].value_counts()

fig = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.3)])
fig.update_layout(title_text="<b>Payment Method Distribution</b>")
fig.show()

fig = px.histogram(df, x="Churn", color="PaymentMethod", title="<b>Customer Payment Method distribution w.r.t. Churn</b>")
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

### Customers using electronic check show the highest churn, while customers using automatic payment methods have the lowest churn.

### 

In [ ]:
 df[df["gender"]=="Male"][["InternetService", "Churn"]].value_counts()

In [ ]:
df[df["gender"]=="Female"][["InternetService", "Churn"]].value_counts()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Bar(
  x = [['Churn:No', 'Churn:No', 'Churn:Yes', 'Churn:Yes'],
       ["Female", "Male", "Female", "Male"]],
  y = [965, 992, 219, 240],
  name = 'DSL',
))

fig.add_trace(go.Bar(
  x = [['Churn:No', 'Churn:No', 'Churn:Yes', 'Churn:Yes'],
       ["Female", "Male", "Female", "Male"]],
  y = [889, 910, 664, 633],
  name = 'Fiber optic',
))

fig.add_trace(go.Bar(
  x = [['Churn:No', 'Churn:No', 'Churn:Yes', 'Churn:Yes'],
       ["Female", "Male", "Female", "Male"]],
  y = [690, 717, 56, 57],
  name = 'No Internet',
))

fig.update_layout(title_text="<b>Churn Distribution w.r.t. Internet Service and Gender</b>")

fig.show()

### The analysis shows that fiber optic customers are the most likely to churn, while DSL and non-internet customers are more stable. Gender does not significantly affect this pattern.

### 

In [ ]:
color_map = {"Yes": "#FF97FF", "No": "#AB63FA"}
fig = px.histogram(df, x="Churn", color="Dependents", barmode="group", title="<b>Dependents distribution</b>", color_discrete_map=color_map)
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

### Customers without dependents show significantly higher churn compared to customers with dependents.

###

In [ ]:
color_map = {"Yes": '#FFA15A', "No": '#00CC96'}
fig = px.histogram(df, x="Churn", color="Partner", barmode="group", title="<b>Chrun distribution w.r.t. Partners</b>", color_discrete_map=color_map)
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

### Customers without partners have a significantly higher churn rate, while customers with partners tend to stay longer.



###

In [ ]:
color_map = {"Yes": '#00CC96', "No": '#B6E880'}
fig = px.histogram(df, x="Churn", color="SeniorCitizen", title="<b>Chrun distribution w.r.t. Senior Citizen</b>", color_discrete_map=color_map)
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

### Senior citizens have a higher churn tendency compared to non-senior customers.

###

In [ ]:
color_map = {"Yes": "#FF97FF", "No": "#AB63FA"}
fig = px.histogram(df, x="Churn", color="OnlineSecurity", barmode="group", title="<b>Churn w.r.t Online Security</b>", color_discrete_map=color_map)
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

### Customers without online security have significantly higher churn, while customers with online security are much more likely to stay.

### 

In [ ]:
color_map = {"Yes": '#FFA15A', "No": '#00CC96'}
fig = px.histogram(df, x="Churn", color="PaperlessBilling",  title="<b>Chrun distribution w.r.t. Paperless Billing</b>", color_discrete_map=color_map)
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

### Customers using paperless billing have a higher churn tendency compared to customers using traditional billing.

### 

In [ ]:
fig = px.histogram(df, x="Churn", color="TechSupport",barmode="group",  title="<b>Chrun distribution w.r.t. TechSupport</b>")
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

### Customers without tech support have significantly higher churn, while customers with tech support are much more likely to stay.

### 

In [ ]:
color_map = {"Yes": '#00CC96', "No": '#B6E880'}
fig = px.histogram(df, x="Churn", color="PhoneService", title="<b>Chrun distribution w.r.t. Phone Service</b>", color_discrete_map=color_map)
fig.update_layout(width=700, height=500, bargap=0.1)
fig.show()

### The churn distribution across phone service users is similar to the overall customer distribution, suggesting that phone service alone does not significantly influence churn behavior.

### 

In [ ]:
sns.set_context("paper",font_scale=1.1)
ax = sns.kdeplot(df.MonthlyCharges[(df["Churn"] == 'No') ],
                color="Red", shade = True);
ax = sns.kdeplot(df.MonthlyCharges[(df["Churn"] == 'Yes') ],
                ax =ax, color="Blue", shade= True);
ax.legend(["Not Churn","Churn"],loc='upper right');
ax.set_ylabel('Density');
ax.set_xlabel('Monthly Charges');
ax.set_title('Distribution of monthly charges by churn');

### The density plot shows that churned customers are concentrated in the higher monthly charge range, indicating price sensitivity is a major driver of churn.

###

In [ ]:
ax = sns.kdeplot(df.TotalCharges[(df["Churn"] == 'No') ],
                color="Gold", shade = True);
ax = sns.kdeplot(df.TotalCharges[(df["Churn"] == 'Yes') ],
                ax =ax, color="Green", shade= True);
ax.legend(["Not Chuurn","Churn"],loc='upper right');
ax.set_ylabel('Density');
ax.set_xlabel('Total Charges');
ax.set_title('Distribution of total charges by churn');

### Customers with lower total charges have a higher churn tendency, indicating that new or low-value customers are more likely to leave.

### 

In [ ]:
fig = px.box(df, x='Churn', y = 'tenure')

# Update yaxis properties
fig.update_yaxes(title_text='Tenure (Months)', row=1, col=1)
# Update xaxis properties
fig.update_xaxes(title_text='Churn', row=1, col=1)

# Update size and title
fig.update_layout(autosize=True, width=750, height=600,
    title_font=dict(size=25, family='Courier'),
    title='<b>Tenure vs Churn</b>',
)

fig.show()

### Customers with low tenure have a much higher churn rate, while long-tenure customers are significantly more loyal.

###

### 

In [ ]:
#Set and compute the Correlation Matrix:
sns.set(style="white")
plt.figure(figsize=(18, 15))

corr = df.apply(lambda x: pd.factorize(x)[0]).corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

ax = sns.heatmap(corr, mask=mask, xticklabels=corr.columns, yticklabels=corr.columns, annot=True, linewidths=.2, cmap='coolwarm', vmin=0.3, vmax=1)

In [ ]:
def distplot(feature, frame, color='r'):
    plt.figure(figsize=(8,3))
    plt.title("Distribution for {}".format(feature))
    ax = sns.distplot(frame[feature], color= color)

In [ ]:
col =  ["tenure", 'MonthlyCharges', 'TotalCharges']
for features in col :distplot(features, df)

###
Tenure: The distribution shows many customers with very low tenure and another group with long tenure, indicating that a large portion of customers are either new or long-term users. This supports earlier findings that churn is higher among new customers.

Monthly Charges: The distribution is spread across a wide range, with a noticeable concentration in the mid-to-high charge levels. This suggests different pricing tiers, and higher charges are linked to increased churn risk.

Total Charges: The distribution is heavily right-skewed, with most customers having low total charges and fewer customers with very high totals. This indicates that many customers leave early, while long-term customers accumulate higher total charges and are more loyal.

### 

# Data preprocessing

In [ ]:
df.columns


In [ ]:
for i in df.columns:
    print(i, ": ", df[i].unique())

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"].map({"No": 0, "Yes": 1})



from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)



In [ ]:
X_train

In [ ]:
X.info()

In [ ]:

num_features = ["tenure", "MonthlyCharges", "TotalCharges"]

cat_ohe = [
           'MultipleLines', 'InternetService', 'OnlineSecurity',
           'OnlineBackup', 'DeviceProtection', 'TechSupport',
           'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod'
          ]

cat_binary = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']




In [ ]:
numeric_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

binary_pipeline = Pipeline([
    ("encoder", OrdinalEncoder())
])

ohe_pipeline = Pipeline([
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_features),
        ("bin", binary_pipeline, cat_binary),
        ("ohe", ohe_pipeline, cat_ohe)
    ]
)

preprocessor.fit_transform(X_train)


In [ ]:
X_train.shape

In [ ]:
X_train

In [ ]:
from sklearn.pipeline import Pipeline

models = []

models.append((
    'Logistic Regression',
    Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(
            solver='liblinear',
            random_state=0,
            class_weight='balanced'
        ))
    ])
))

models.append((
    'SVC',
    Pipeline([
        ("preprocess", preprocessor),
        ("model", SVC(kernel='linear', probability=True, random_state=0))
    ])
))

models.append((
    'Kernel SVM',
    Pipeline([
        ("preprocess", preprocessor),
        ("model", SVC(kernel='rbf', probability=True, random_state=0))
    ])
))

models.append((
    'KNN',
    Pipeline([
        ("preprocess", preprocessor),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ])
))

models.append((
    'Gaussian NB',
    Pipeline([
        ("preprocess", preprocessor),
        ("model", GaussianNB())
    ])
))

models.append((
    'Decision Tree',
    Pipeline([
        ("preprocess", preprocessor),
        ("model", DecisionTreeClassifier(criterion='entropy', random_state=0))
    ])
))

models.append((
    'Random Forest',
    Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(n_estimators=100, random_state=0))
    ])
))

models.append((
    'Adaboost',
    Pipeline([
        ("preprocess", preprocessor),
        ("model", AdaBoostClassifier())
    ])
))

models.append((
    'Gradient Boost',
    Pipeline([
        ("preprocess", preprocessor),
        ("model", GradientBoostingClassifier())
    ])
))

models.append((
    'Voting Classifier',
    Pipeline([
        ("preprocess", preprocessor),
        ("model", VotingClassifier(
            estimators=[
                ('gbc', GradientBoostingClassifier()),
                ('lr', LogisticRegression()),
                ('abc', AdaBoostClassifier())
            ],
            voting='soft'
        ))
    ])
))


In [ ]:


model_results = []

for name, model in models:
    kfold = model_selection.KFold(n_splits=10, shuffle=True, random_state=42)

    cv_acc = model_selection.cross_val_score(
        model, X_train, y_train, cv=kfold, scoring="accuracy"
    )

    cv_auc = model_selection.cross_val_score(
        model, X_train, y_train, cv=kfold, scoring="roc_auc"
    )

    model_results.append({
        "Algorithm": name,
        "ROC AUC Mean": round(cv_auc.mean() * 100, 2),
        "ROC AUC STD": round(cv_auc.std() * 100, 2),
        "Accuracy Mean": round(cv_acc.mean() * 100, 2),
        "Accuracy STD": round(cv_acc.std() * 100, 2),
    })

model_results = (
    pd.DataFrame(model_results)
    .sort_values(by="ROC AUC Mean", ascending=False)
    .reset_index(drop=True)
)


In [ ]:
model_results

In [ ]:
fig = plt.figure(figsize=(25,15))
ax = fig.add_subplot(111)
plt.boxplot(acc_results)
ax.set_xticklabels(names)

plt.ylabel('ROC AUC Score\n',
horizontalalignment="center",fontstyle = "normal", 
fontsize = "large", fontfamily = "sans-serif")

plt.xlabel('\n Baseline Classification Algorithms\n',
horizontalalignment="center",fontstyle = "normal", 
fontsize = "large", fontfamily = "sans-serif")

plt.title('Accuracy Score Comparison \n',
horizontalalignment="center", fontstyle = "normal", 
fontsize = "22", fontfamily = "sans-serif")

plt.xticks(rotation=0, horizontalalignment="center")
plt.yticks(rotation=0, horizontalalignment="right")
plt.show()



In [ ]:
fig = plt.figure(figsize=(25,15))
ax = fig.add_subplot(111)
plt.boxplot(auc_results)
ax.set_xticklabels(names)

plt.ylabel('ROC AUC Score\n',
horizontalalignment="center",fontstyle = "normal", 
fontsize = "large", fontfamily = "sans-serif")

plt.xlabel('\n Baseline Classification Algorithms\n',
horizontalalignment="center",fontstyle = "normal", 
fontsize = "large", fontfamily = "sans-serif")

plt.title('ROC AUC Comparison \n',
horizontalalignment="center", fontstyle = "normal", 
fontsize = "22", fontfamily = "sans-serif")

plt.xticks(rotation=0, horizontalalignment="center")
plt.yticks(rotation=0, horizontalalignment="right")
plt.show()

In [ ]:
#evaluation of results
def model_evaluation(y_test, y_pred, model_name):
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)  
    f2 = fbeta_score(y_test, y_pred, beta = 2.0)

    results = pd.DataFrame([[model_name, acc, prec, rec, f1, f2]], 
                       columns = ["Model", "Accuracy", "Precision", "Recall",
                                 "F1 SCore", "F2 Score"])
    results = results.sort_values(["Precision", "Recall", "F2 Score"], ascending = False)
    return results

In [ ]:
from sklearn.pipeline import Pipeline

# Logistic regression
classifier = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(random_state=0))
])
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)


# SVC
classifier2 = Pipeline([
    ("preprocess", preprocessor),
    ("model", SVC(kernel='linear', random_state=0))
])
classifier2.fit(X_train, y_train)
y_pred2 = classifier2.predict(X_test)


# KNN
classifier3 = Pipeline([
    ("preprocess", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=22))
])
classifier3.fit(X_train, y_train)
y_pred3 = classifier3.predict(X_test)


# Kernel SVM
classifier4 = Pipeline([
    ("preprocess", preprocessor),
    ("model", SVC(kernel="rbf", random_state=0))
])
classifier4.fit(X_train, y_train)
y_pred4 = classifier4.predict(X_test)


# Naive Bayes
classifier5 = Pipeline([
    ("preprocess", preprocessor),
    ("model", GaussianNB())
])
classifier5.fit(X_train, y_train)
y_pred5 = classifier5.predict(X_test)


# Decision tree
classifier6 = Pipeline([
    ("preprocess", preprocessor),
    ("model", DecisionTreeClassifier(criterion="entropy", random_state=0))
])
classifier6.fit(X_train, y_train)
y_pred6 = classifier6.predict(X_test)


# Random Forest
classifier7 = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(n_estimators=72, random_state=0))
])
classifier7.fit(X_train, y_train)
y_pred7 = classifier7.predict(X_test)


# AdaBoost
classifier8 = Pipeline([
    ("preprocess", preprocessor),
    ("model", AdaBoostClassifier())
])
classifier8.fit(X_train, y_train)
y_pred8 = classifier8.predict(X_test)


# Gradient Boost
classifier9 = Pipeline([
    ("preprocess", preprocessor),
    ("model", GradientBoostingClassifier())
])
classifier9.fit(X_train, y_train)
y_pred9 = classifier9.predict(X_test)


# Voting Classifier
classifier10 = Pipeline([
    ("preprocess", preprocessor),
    ("model", VotingClassifier(
        estimators=[
            ('gbc', GradientBoostingClassifier()),
            ('lr', LogisticRegression()),
            ('abc', AdaBoostClassifier())
        ],
        voting='soft'
    ))
])
classifier10.fit(X_train, y_train)
y_pred10 = classifier10.predict(X_test)


In [ ]:
lr = model_evaluation(y_test, y_pred, "Logistic Regression")
svm = model_evaluation(y_test, y_pred2, "SVM (Linear)")
knn = model_evaluation(y_test, y_pred3, "K-Nearest Neighbours")
k_svm = model_evaluation(y_test, y_pred4, "Kernel SVM")
nb = model_evaluation(y_test, y_pred5, "Naive Bayes")
dt = model_evaluation(y_test, y_pred6, "Decision Tree")
rf = model_evaluation(y_test, y_pred7, "Random Forest")
ab = model_evaluation(y_test, y_pred8, "Adaboost")
gb = model_evaluation(y_test, y_pred9, "Gradient Boost")
vc = model_evaluation(y_test, y_pred10, "Voting Classifier")

In [ ]:

eval_ = (
    pd.concat([lr, svm, knn, k_svm, nb, dt, rf, ab, gb, vc])
      .sort_values(["F1 SCore"], ascending=False)
      .reset_index(drop=True)
)

eval_


In [ ]:
predictions = [y_pred, y_pred2 , y_pred3, y_pred4, y_pred5, y_pred5, y_pred6, y_pred7,
              y_pred8, y_pred9, y_pred10]

for i, j in zip(predictions, eval_.Model.values):
    plt.figure(figsize=(4,3))
    sns.heatmap(confusion_matrix(y_test, i),
                annot=True,fmt = "d",linecolor="k",linewidths=3)
    
    plt.title(j,fontsize=14)
    plt.show()

In [ ]:
def k_fold_cross_validation(classifier_name, name):
    accuracies = cross_val_score(estimator=classifier_name,
                            X=X_train, y=y_train, cv =10)
    print(name, "accuracy: %0.2f (+/- %0.2f)"  % (accuracies.mean(), accuracies.std() * 2))

In [ ]:
k_fold_cross_validation(classifier8, "Adaboost")

In [ ]:
k_fold_cross_validation(classifier10, "Voting classifier")

In [ ]:
k_fold_cross_validation(classifier9, "Gradient Boost classifier")

In [ ]:
k_fold_cross_validation(classifier, "Logistic regression")

In [ ]:
k_fold_cross_validation(classifier4, "Kernel SVM")

In [ ]:
k_fold_cross_validation(classifier7, "Random Forest")

In [ ]:
k_fold_cross_validation(classifier5, "GaussianNB")

In [ ]:
k_fold_cross_validation(classifier2, "SVC")

In [ ]:
k_fold_cross_validation(classifier3, "KNeighborsClassifier")

In [ ]:
k_fold_cross_validation(classifier6, "DecisionTreeClassifier")

In [ ]:
# ROC Curve



from sklearn.metrics import roc_curve, roc_auc_score

def ROC_curve(classifier_, name, y_pred_):
    classifier_.fit(X_train, y_train)

    # Handle models with or without predict_proba
    if hasattr(classifier_, "predict_proba"):
        probs = classifier_.predict_proba(X_test)[:, 1]
    else:
        probs = classifier_.decision_function(X_test)

    classifier_roc_auc = roc_auc_score(y_test, probs)
    fpr, tpr, thresholds = roc_curve(y_test, probs)

    plt.figure(figsize=(14, 6))

    label_ = f"{name} (AUC = {classifier_roc_auc:.2f})"

    # Plot ROC curve
    plt.plot(fpr, tpr, label=label_)
    plt.plot([0, 1], [0, 1], 'k--', label='Base Rate')

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])

    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curve', fontsize=20)

    plt.legend(loc="lower right")
    plt.show()


In [ ]:
preds = [y_pred, y_pred2, y_pred3, y_pred4, y_pred5, y_pred6, y_pred7,
              y_pred8, y_pred9, y_pred10]
classifiers = [classifier , classifier2, classifier3, classifier4, classifier5, classifier6, classifier7,
             classifier8, classifier9, classifier10]
model_names_ = ["Logistic Regression",  "SVC", "K-Nearest Neighbours", "Kernel SVM", "Naive Bayes",
               "Decision Tree", "Random Forest", "Adaboost", "Gradient Boost",  "Voting Classifier"]

for i, j, k in zip(classifiers, model_names_, predictions):
    ROC_curve(i, j, k) 

In [ ]:
# # Cross validation

# from sklearn.model_selection import cross_val_score

# # Function that will track the mean value and the standard deviation of the accuracy
# def cvDictGen(functions, scr, X_train = X, y_train = y, cv = 10):
#     cvDict = {}
#     for func in functions:
#         cvScore = cross_val_score(func, X_train, y_train, cv = cv, scoring = scr)
#         cvDict[str(func).split('(')[0]] = [cvScore.mean(), cvScore.std()]
    
#     return cvDict


classifiers = [
    ("LogisticRegression", classifier),
    ("SVC_Linear", classifier2),
    ("KNeighborsClassifier", classifier3),
    ("SVC_RBF", classifier4),
    ("GaussianNB", classifier5),
    ("DecisionTreeClassifier", classifier6),
    ("RandomForestClassifier", classifier7),
    ("AdaBoostClassifier", classifier8),
    ("GradientBoostingClassifier", classifier9),
    ("VotingClassifier", classifier10)
]


def cvDictGen(functions, scr, X_train=X, y_train=y, cv=10):
    cvDict = {}
    for name, func in functions:
        cvScore = cross_val_score(func, X_train, y_train, cv=cv, scoring=scr)
        cvDict[name] = [cvScore.mean(), cvScore.std()]
    return cvDict


In [ ]:
cvD = cvDictGen(classifiers, scr = 'roc_auc')
cvD

###
###


### Based on the ROC–AUC scores, I shortlisted the top four performing models for hyperparameter tuning

### AdaBoostClassifier
### GradientBoostingClassifier
### VotingClassifier
### LogisticRegression



### 

In [ ]:
# AdaBoostClassifier

ada_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", AdaBoostClassifier(n_estimators=100, learning_rate=0.3))
])


In [ ]:
# GradientBoostingClassifier

gb_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", GradientBoostingClassifier(n_estimators=300, max_depth=1))
])


In [ ]:
# VotingClassifier


voting_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", VotingClassifier(
        estimators=[
            ("gb", GradientBoostingClassifier()),
            ("lr", LogisticRegression(max_iter=1000)),
            ("ada", AdaBoostClassifier())
        ],
        voting="soft"
    ))
])


In [ ]:
# logistic Regression


log_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
ada_pipeline.fit(X_train, y_train)
gb_pipeline.fit(X_train, y_train)
voting_pipeline.fit(X_train, y_train)
log_pipeline.fit(X_train, y_train)


# Hyperparameter tuning using GridSearchCV

In [ ]:
# For AdaBoostClassifier

from sklearn.model_selection import GridSearchCV

ada_param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.01, 0.1, 0.3, 1.0]
}

ada_gs = GridSearchCV(
    estimator=ada_pipeline,
    param_grid=ada_param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

ada_gs.fit(X_train, y_train)

ada_best_pipeline_gs = ada_gs.best_estimator_

print("AdaBoost best params:", ada_gs.best_params_)
print("AdaBoost best ROC-AUC:", ada_gs.best_score_)


In [ ]:
# For GradientBoostingClassifier



gb_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__learning_rate": [0.05, 0.1, 0.2],
    "model__max_depth": [1, 2, 3]
}



gb_gs = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

gb_gs.fit(X_train, y_train)

gb_best_pipeline_gs = gb_gs.best_estimator_

print("Gradient Boost best params:", gb_gs.best_params_)
print("Gradient Boost best ROC-AUC:", gb_gs.best_score_)



In [ ]:
# For Voting Classifier

voting_param_grid = {
    "model__weights": [
        [1, 1, 1],
        [2, 1, 1],
        [1, 2, 1],
        [1, 1, 2]
    ],
    "model__gb__n_estimators": [100, 200],
    "model__gb__learning_rate": [0.05, 0.1],
    "model__ada__n_estimators": [50, 100]
}

voting_gs = GridSearchCV(
    estimator=voting_pipeline,
    param_grid=voting_param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

voting_gs.fit(X_train, y_train)

voting_best_pipeline_gs = voting_gs.best_estimator_

print("Voting best params:", voting_gs.best_params_)
print("Voting best ROC-AUC:", voting_gs.best_score_)


In [ ]:
# For Logistic Regression

log_param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"],
    "model__class_weight": [None, "balanced"]
}


from sklearn.model_selection import GridSearchCV

log_gs = GridSearchCV(
    estimator=log_pipeline,
    param_grid=log_param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

log_gs.fit(X_train, y_train)

log_best_pipeline_gs = log_gs.best_estimator_

print("Logistic Regression best params:", log_gs.best_params_)
print("Logistic Regression best ROC-AUC:", log_gs.best_score_)


# Hyperparameter tuning using Randomized search CV

In [ ]:
# For AdaBoostClassifier

from sklearn.model_selection import RandomizedSearchCV

ada_param_dist = {
    "model__n_estimators": [50, 100, 150, 200, 250, 300],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]
}

ada_rs = RandomizedSearchCV(
    estimator=ada_pipeline,
    param_distributions=ada_param_dist,
    n_iter=10,              # number of random combinations
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

ada_rs.fit(X_train, y_train)

ada_best_pipeline_rs = ada_rs.best_estimator_

print("AdaBoost best params:", ada_rs.best_params_)
print("AdaBoost best ROC-AUC:", ada_rs.best_score_)


In [ ]:
# For GradientBoostingClassifier


gb_param_dist = {
    "model__n_estimators": [100, 150, 200, 250, 300],
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "model__max_depth": [1, 2, 3, 4]
}

gb_rs = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=gb_param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

gb_rs.fit(X_train, y_train)

gb_best_pipeline_rs = gb_rs.best_estimator_

print("Gradient Boost best params:", gb_rs.best_params_)
print("Gradient Boost best ROC-AUC:", gb_rs.best_score_)

In [ ]:
# For Voting Classifier


voting_param_dist = {
    "model__gb__n_estimators": [100, 150, 200],
    "model__gb__learning_rate": [0.05, 0.1, 0.2],
    "model__lr__C": [0.1, 1, 10],
    "model__ada__n_estimators": [50, 100, 150]
}

voting_rs = RandomizedSearchCV(
    estimator=voting_pipeline,
    param_distributions=voting_param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

voting_rs.fit(X_train, y_train)

voting_best_pipeline_rs = voting_rs.best_estimator_

print("Voting best params:", voting_rs.best_params_)
print("Voting best ROC-AUC:", voting_rs.best_score_)


In [ ]:
# For Logistic Regression


log_param_dist = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"],
    "model__class_weight": [None, "balanced"]
}

log_rs = RandomizedSearchCV(
    estimator=log_pipeline,
    param_distributions=log_param_dist,
    n_iter=10,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

log_rs.fit(X_train, y_train)

log_best_pipeline_rs = log_rs.best_estimator_

print("Logistic Regression best params:", log_rs.best_params_)
print("Logistic Regression best ROC-AUC:", log_rs.best_score_)


In [ ]:
from sklearn.metrics import roc_auc_score

tuned_pipelines = {
    "AdaBoost_Tuned_gs": ada_best_pipeline_gs,
    "GradientBoost_Tuned_gs": gb_best_pipeline_gs,
    "Voting_Tuned_gs": voting_best_pipeline_gs,
    "AdaBoost_Tuned_rs": ada_best_pipeline_rs,
    "GradientBoost_Tuned_rs": gb_best_pipeline_rs,
    "Voting_Tuned_rs": voting_best_pipeline_rs
    
}

print("\n📊 Tuned model performance on test set\n")

for name, pipe in tuned_pipelines.items():
    y_prob = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    print(f"{name}: ROC-AUC = {auc:.4f}")


In [ ]:
# AdaBoost best ROC-AUC: 0.8463845134807778
# Gradient Boost best ROC-AUC: 0.8495862193307007
# Voting best ROC-AUC: 0.8496038820438855
# Logistic Regression best ROC-AUC: 0.846044743013311

# AdaBoost best ROC-AUC: 0.8463845134807778
# Gradient Boost best ROC-AUC: 0.8484674676785
# Voting best ROC-AUC: 0.8495247590670727
# Logistic Regression best ROC-AUC: 0.8460608901608893

In [ ]:
import joblib

joblib.dump(voting_best_pipeline_gs, "final_churn_model.pkl")
print("Model saved successfully.")




In [ ]:
model = joblib.load("final_churn_model.pkl")


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    fbeta_score
)

# Predictions
y_pred = voting_best_pipeline_gs.predict(X_test)
y_prob = voting_best_pipeline_gs.predict_proba(X_test)[:, 1]

# Metrics
cm = confusion_matrix(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
f2 = fbeta_score(y_test, y_pred, beta=2)
roc_auc = roc_auc_score(y_test, y_prob)

print("Confusion Matrix:\n", cm)
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))
print("F2 Score:", round(f2, 4))
print("ROC-AUC:", round(roc_auc, 4))




In [ ]:
sample = X_test.iloc[:1]
prediction = model.predict(sample)
probability = model.predict_proba(sample)[:, 1]

print("Prediction:", prediction[0])
print("Churn probability:", probability[0])


In [ ]:
sample

In [ ]:
df.iloc[437:438]

Customer Churn Prediction – Project Description

Developed a machine learning model to predict telecom customer churn using customer demographic, billing, and service usage data.

Performed exploratory data analysis (EDA) on a dataset of 7043 customers and 21 features to identify key factors influencing churn. 



Identified important churn drivers such as contract type, tenure, monthly charges, payment method, and internet service type.

Cleaned and preprocessed the dataset by handling missing values, converting data types, and removing irrelevant features like customerID.

Implemented feature engineering and data preprocessing pipelines using ColumnTransformer, StandardScaler, OrdinalEncoder, and OneHotEncoder.

Split the dataset into training and testing sets (80/20) for model development and evaluation. 



Trained and compared multiple machine learning algorithms including:

Logistic Regression

Support Vector Machine (SVM)

K-Nearest Neighbors

Decision Tree

Random Forest

AdaBoost

Gradient Boosting

Naive Bayes

Voting Classifier Ensemble

Evaluated model performance using Accuracy, Precision, Recall, F1 Score, ROC-AUC, and Confusion Matrix.

Applied 10-fold cross-validation to ensure model stability and reliability.

Achieved approximately 80% prediction accuracy, with ensemble models such as Voting Classifier and Gradient Boosting performing the best. 


Visualized model performance using ROC curves, confusion matrices, and comparison charts to interpret model results.